# Myllia: Direction Notebook (Bilinear Conditional Factorization)

This notebook implements a medium-large architecture shift:

- Learn a low-rank interaction between perturbed gene embeddings and output gene embeddings
- Predict the full delta vector as a bilinear form with rank `R`
- Train with a metric-aligned weighted L1 proxy and evaluate with the official `myllia_score`

Core model:
\[
\hat D_{i,j} = \langle W_p z_{g_i},\; W_o u_j \rangle + b_j + b_i
\]
where:
- `z_{g_i}` is an embedding for the perturbed gene `g_i`
- `u_j` is an embedding for output gene `j`
- `R` is a small rank (16 to 64)

This uses `training_cells.h5ad` to build output gene embeddings.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

import anndata as ad
import scanpy as sc
from scipy import sparse

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

from myllia_metric import myllia_score

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

EMB_DIM_PERT = 128   # pert gene embedding dim (from SVD)
EMB_DIM_OUT = 128   # output gene embedding dim (from SVD)
RANK_R = 32    # low-rank interaction size

DROPOUT = 0.10
LR = 2e-3
WD = 1e-4
EPOCHS = 400
BATCH_GENES = 16 # minibatch over perturbed genes
EVAL_EVERY = 25
PATIENCE = 12 # early stopping patience in eval steps

# Gate parameters should match metric structure
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

ROOT = Path(".")

# --- added: CV alpha sweep + refit ---
MODEL_SEEDS = [6, 7, 8]
ALPHA_GRID = np.linspace(0.0, 0.7, 36).astype(np.float32)
GRAD_CLIP = 1.0
H5AD_PATH = ROOT / "data" / "training_cells.h5ad"  # used ONLY for perts not in gene_columns


In [2]:
def score_delta(dt, dp):
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)
    return {
        "score": float(r.score),
        "wcos": float(r.wcos),
        "mean_term": float(r.mean_term),
        "pred_wmae": float(r.pred_wmae),
    }


In [3]:
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :] # (80, 5127) delta vs non-targeting

delta_baseline = D_train.mean(axis=0).astype(np.float32)

# pert_id -> gene symbol for leaderboard (first 60)
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))


Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


In [4]:
gt_df = pd.read_csv('Data/training_data_ground_truth_table.csv')
baseline_wmae_t = torch.tensor(
    gt_df["baseline_wmae"].to_numpy(dtype=np.float32),
    device=device,
    dtype=torch.float32,
)


Build gene embeddings from D_train (SVD on signed-log transformed deltas). Build embeddings for output genes (gene_columns) via SVD on (80, 5127). Pert embeddings are looked up by gene symbol in gene_columns, otherwise fallback to the mean embedding.

In [5]:
val_targets = df_valmap["pert"].astype(str).tolist()

union_genes = sorted(set([g.upper() for g in gene_columns] +
                         [g.upper() for g in train_genes.tolist()] +
                         [g.upper() for g in val_targets]))

geneU = pd.Index([str(g).upper() for g in gene_columns])

# (80, 5127) dense -> signed log transform to handle negatives
X = D_train.astype(np.float32, copy=True)
X = np.sign(X) * np.log2(1.0 + np.abs(X))

# SVD across genes: components_ is (k, 5127) so transpose to (5127, k)
svd = TruncatedSVD(n_components=max(EMB_DIM_PERT, EMB_DIM_OUT), random_state=SEED)
svd.fit(X)

gene_emb_all = svd.components_.T.astype(np.float32)  # (G, k)
k_svd = gene_emb_all.shape[1]
print("SVD k:", k_svd, "gene_emb_all:", gene_emb_all.shape)

# Map output genes to embeddings
gene2emb_pert = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_PERT].copy()
                 for i in range(len(gene_columns))}
gene2emb_out  = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_OUT ].copy()
                 for i in range(len(gene_columns))}

emb_fallback_pert = gene_emb_all[:, :EMB_DIM_PERT].mean(axis=0).astype(np.float32)
emb_fallback_out  = gene_emb_all[:, :EMB_DIM_OUT ].mean(axis=0).astype(np.float32)

missing_emb_pert = {}

def emb_pert(g: str) -> np.ndarray:
    gU = str(g).upper()
    if gU in gene2emb_pert:
        return gene2emb_pert[gU]
    if gU in missing_emb_pert:
        return missing_emb_pert[gU]
    return emb_fallback_pert

def emb_out(g: str) -> np.ndarray:
    return gene2emb_out.get(str(g).upper(), emb_fallback_out)

# Output gene embeddings in the exact output gene order
U_out = np.vstack([emb_out(g) for g in gene_columns]).astype(np.float32)    # (G, d_out)

# Pert embeddings for your 80 training perts (fallback if not in gene_columns)
Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)  # (80, d_pert)

print("U_out:", U_out.shape, "Z_train:", Z_train.shape)

# Optional: visibility into coverage
missing_train = [g for g in train_genes.tolist() if str(g).upper() not in geneU]
missing_val = [g for g in val_targets if str(g).upper() not in geneU]
if missing_train:
    print(f"train perts not in gene_columns: {len(missing_train)}. Example: {missing_train[:12]}")
if missing_val:
    print(f"val perts not in gene_columns: {len(missing_val)}. Example: {missing_val[:12]}")



# -----------------------------
# h5ad ONLY for missing perts: embed by control-cell coexpression
# -----------------------------
missing_all = sorted(set([str(x).upper() for x in (missing_train + missing_val)]))

if len(missing_all) > 0:
    try:
        import scipy.sparse as sp
        import anndata as ad

        print("[h5ad] building embeddings for missing perts:", len(missing_all))
        adata = ad.read_h5ad(str(H5AD_PATH))

        # pick perturbation column
        pert_col = None
        for c in ["sgrna_symbol", "pert_symbol", "pert", "perturbation", "gene", "target_gene"]:
            if c in adata.obs.columns:
                pert_col = c
                break
        if pert_col is None:
            raise ValueError("Could not find perturbation column in h5ad obs.")

        Xc = adata.X
        if not sp.issparse(Xc):
            Xc = sp.csr_matrix(Xc)
        else:
            Xc = Xc.tocsr()

        # normalize ALL genes: CPM10K then log2(1+x)
        cell_sum = np.asarray(Xc.sum(axis=1)).ravel().astype(np.float64)
        scale = (10000.0 / cell_sum).astype(np.float64)

        Xn = Xc.multiply(scale[:, None]).tocsr()
        Xn.data = np.log1p(Xn.data) / np.log(2.0)

        ctrl_mask = (adata.obs[pert_col].astype(str).to_numpy() == "non-targeting")
        if int(ctrl_mask.sum()) == 0:
            raise ValueError("No non-targeting control cells found in h5ad.")

        var = {str(g).upper(): i for i, g in enumerate(adata.var_names.astype(str).to_numpy())}

        # indices for the 5127 output genes in the 19,226 gene space
        out_idx = np.array([var[str(g).upper()] for g in gene_columns if str(g).upper() in var], dtype=np.int64)
        if len(out_idx) != len(gene_columns):
            miss_out = [g for g in gene_columns if str(g).upper() not in var]
            raise ValueError(f"{len(miss_out)} output genes missing from h5ad var_names (unexpected). Example: {miss_out[:10]}")

        Xout = Xn[ctrl_mask][:, out_idx]  # (n_ctrl, 5127)
        if sp.issparse(Xout):
            Xout = Xout.toarray()
        Xout = Xout.astype(np.float32)

        mu = Xout.mean(axis=0, keepdims=True)
        sd = Xout.std(axis=0, keepdims=True) + 1e-6
        Xout_z = (Xout - mu) / sd

        # use the *existing* pert embedding space (from SVD on D_train): (5127, d_pert)
        P_out = gene_emb_all[:, :EMB_DIM_PERT].astype(np.float32)

        topk = 256
        made = 0
        for gU in missing_all:
            if gU not in var:
                continue

            xg = Xn[ctrl_mask, var[gU]]
            if sp.issparse(xg):
                xg = xg.toarray()
            xg = np.asarray(xg).ravel().astype(np.float32)

            xg = (xg - xg.mean()) / (xg.std() + 1e-6)

            corr = (xg[:, None] * Xout_z).mean(axis=0)  # (5127,)
            idx = np.argsort(-np.abs(corr))[:topk]
            w = corr[idx].astype(np.float32)

            z = (w[:, None] * P_out[idx]).sum(axis=0)
            z = z / (np.linalg.norm(z) + 1e-12)

            missing_emb_pert[gU] = z.astype(np.float32)
            made += 1

        print("[h5ad] embedded missing perts:", made, "of", len(missing_all))
    except Exception as e:
        print("[h5ad] skipped missing pert embeddings due to error:", repr(e))


SVD k: 80 gene_emb_all: (5127, 80)
U_out: (5127, 80) Z_train: (80, 80)
train perts not in gene_columns: 8. Example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']
[h5ad] building embeddings for missing perts: 16
[h5ad] embedded missing perts: 16 of 16


In [6]:
def gate_smoothstep(x, a = GATE_A, b = GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def per_row_weighted_l1_like(delta_true: torch.Tensor, delta_pred: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    """
    Same logic as weighted_l1_like, but returns (N,) per-row.
    Assumes your existing gate_smoothstep(...) exists.
    """
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)  # (N, G)
    err = torch.abs(delta_pred - delta_true)                        # (N, G)
    num = torch.sum(w * err, dim=1)                                 # (N,)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)                 # (N,)
    return num / den                                                # (N,)

import torch

def weighted_l1_like_rowweighted(
    delta_true: torch.Tensor,     # (N, G)
    delta_pred: torch.Tensor,     # (N, G)
    baseline_wmae: torch.Tensor,  # (N,)
    *,
    eps: float = 1e-8,
    mode: str = "inv_sqrt",       # "inv", "inv_sqrt", "inv_log"
    clamp_min: float = 0.5,
    clamp_max: float = 3.0,
) -> torch.Tensor:
    """
    Row-weighted version of your existing weighted_l1_like.

    Weight idea:
      - baseline_wmae small => ratio metric is unforgiving => upweight that row
      - baseline_wmae large => easier => downweight a bit

    Returns: scalar loss
    """
    # per-row unweighted loss from your current function
    per_row = per_row_weighted_l1_like(delta_true, delta_pred, eps=eps)  # (N,)

    b = baseline_wmae.to(delta_true.device).to(delta_true.dtype)

    if mode == "inv":
        w = 1.0 / (b + eps)
    elif mode == "inv_sqrt":
        w = 1.0 / torch.sqrt(b + eps)
    elif mode == "inv_log":
        w = 1.0 / torch.log1p(b + eps)
    else:
        raise ValueError(f"Unknown mode={mode}")

    # clamp to prevent a few rows from dominating training
    w = torch.clamp(w, min=clamp_min, max=clamp_max)

    # normalized weighted mean (stable)
    return torch.sum(w * per_row) / torch.clamp(torch.sum(w), min=eps)

In [7]:
class BilinearDeltaModel(nn.Module):
    def __init__(self, d_pert, d_out, rank_r, dropout):
        super().__init__()
        self.rank_r = rank_r

        self.proj_p = nn.Sequential(
            nn.Linear(d_pert, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.proj_o = nn.Sequential(
            nn.Linear(d_out, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # biases
        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None  # set via set_gene_bias

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = torch.nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = torch.nn.Parameter(torch.zeros(1, device=dev))

    def forward(self, z_pert, u_out):
        p = self.proj_p(z_pert)    # (B, R)
        o = self.proj_o(u_out)     # (G, R)
        y = p @ o.T                # (B, G)
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

In [8]:
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

Uo_t = torch.tensor(U_out, device=device) # (G, d_out)
Zt = torch.tensor(Z_train, device=device) # (N, d_pert)
Yt = torch.tensor(Y, device=device) # (N, G)

print("N:", N, "G:", G, "device:", device)

N: 80 G: 5127 device: cuda


In [27]:
DROPOUT = 0.10
LR = 2e-3 * 0.6
WD = 1e-4
EPOCHS = 400
BATCH_GENES = 16 # minibatch over perturbed genes
EVAL_EVERY = 25
PATIENCE = 12 # early stopping patience in eval steps

# Gate parameters should match metric structure
GATE_A = 0.0
GATE_B = 0.20
EPS = 1e-12

ALPHA_GRID = [0.778]

def apply_shrink(pred, baseline, alpha):
    # pred: (B,G) ; baseline: (G,)
    return float(alpha) * pred + (1.0 - float(alpha)) * baseline[None, :]

def train_one_fold(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_state = None
    best_va_pred = None  # unshrunk predictions at best checkpoint
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            dt_b = Yt.index_select(0, b_t)                 # (B, G)
            dp_b = pred                                    # (B, G)
            bw_b = baseline_wmae_t.index_select(0, b_t)     # (B,)

            loss = weighted_l1_like_rowweighted(
                dt_b, dp_b, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % 1 == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo_t).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]

            # alpha sweep (0..0.7) on this fold
            sc_best = -1e18
            a_best = 0.0
            for a in ALPHA_GRID:
                pred_a = apply_shrink(va_pred, delta_baseline, float(a))
                sc = score_delta(va_true, pred_a)["score"]
                if sc > sc_best:
                    sc_best = sc
                    a_best = float(a)

            if sc_best > best_score:
                best_score = sc_best
                best_alpha = a_best
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_va_pred = va_pred.copy()
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_state, best_va_pred

# --- CV: collect OOF preds and optimize a GLOBAL alpha on OOF only ---
kf = KFold(n_splits=8, shuffle=True, random_state=SEED)

oof_pred = np.zeros_like(Y, dtype=np.float32)
oof_hit = np.zeros((N,), dtype=np.int32)

fold_scores = []
fold_alphas = []
fold_epochs = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
    best_score, best_alpha, best_epoch, best_state, best_va_pred = train_one_fold(tr_idx, va_idx, seed=SEED)
    fold_scores.append(float(best_score))
    fold_alphas.append(float(best_alpha))
    fold_epochs.append(int(best_epoch))

    oof_pred[va_idx] = best_va_pred
    oof_hit[va_idx] += 1

    print(f"fold {fold}: best_score={best_score:.6f} best_alpha={best_alpha:.3f} best_epoch={best_epoch}")

if not np.all(oof_hit == 1):
    print("[warn] OOF coverage not 1 everywhere. min/max:", int(oof_hit.min()), int(oof_hit.max()))

print("cv mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))

EPOCHS_MED = int(np.median(fold_epochs))
print("median best_epoch =", EPOCHS_MED)

# Global alpha chosen on OOF predictions only (more stable than per-fold alpha roulette)
best_global_alpha = 0.0
best_global_score = -1e18
for a in ALPHA_GRID:
    pred_a = apply_shrink(oof_pred, delta_baseline, float(a))
    sc = score_delta(Y, pred_a)["score"]
    if sc > best_global_score:
        best_global_score = sc
        best_global_alpha = float(a)

print("OOF global alpha:", best_global_alpha, "OOF score:", best_global_score)

ALPHA_SHRINK = best_global_alpha


fold 1: best_score=0.166176 best_alpha=0.778 best_epoch=31
fold 2: best_score=0.114720 best_alpha=0.778 best_epoch=28
fold 3: best_score=0.113649 best_alpha=0.778 best_epoch=25
fold 4: best_score=0.122409 best_alpha=0.778 best_epoch=43
fold 5: best_score=0.158052 best_alpha=0.778 best_epoch=26
fold 6: best_score=0.191780 best_alpha=0.778 best_epoch=24
fold 7: best_score=0.182183 best_alpha=0.778 best_epoch=16
fold 8: best_score=0.129139 best_alpha=0.778 best_epoch=18
cv mean: 0.14726350865234605 std: 0.0291769675525368
median best_epoch = 25
OOF global alpha: 0.778 OOF score: 0.14603729934735732


In [36]:
import numpy as np
import pandas as pd

def smoothstep(t):
    return t * t * (3.0 - 2.0 * t)

def gate_smoothstep_np(x, a=0.0, b=0.2):
    x = np.asarray(x, dtype=np.float32)
    t = (x - a) / (b - a)
    t = np.clip(t, 0.0, 1.0)
    return smoothstep(t).astype(np.float32)

def weighted_cosine_np(dt, dp, w, eps=1e-12):
    wa = w * dt
    wb = w * dp
    num = np.sum(wa * wb, axis=1)
    da  = np.sqrt(np.sum(wa * wa, axis=1))
    db  = np.sqrt(np.sum(wb * wb, axis=1))
    denom = np.maximum(da * db, eps)
    return (num / denom).astype(np.float32)

def weighted_mae_np(err, w, eps=1e-12):
    num = np.sum(w * np.abs(err), axis=1)
    den = np.maximum(np.sum(w, axis=1), eps)
    return (num / den).astype(np.float32)

def metric_components_per_row(
    delta_true,          # (N,G)
    delta_pred,          # (N,G)
    *,
    gene_gate_a=0.0,
    gene_gate_b=0.2,
    ratio_gate_a=0.0,
    ratio_gate_b=0.2,
    log_clip=4.0,
    eps=1e-12,
    baseline_wmae_per=None,  # optional (N,) override denominator
):
    dt = np.asarray(delta_true, dtype=np.float32)
    dp = np.asarray(delta_pred, dtype=np.float32)
    assert dt.shape == dp.shape and dt.ndim == 2

    w_gene = gate_smoothstep_np(np.abs(dt), a=gene_gate_a, b=gene_gate_b)

    wcos_per = weighted_cosine_np(dt, dp, w_gene, eps=eps)

    pred_wmae_per = weighted_mae_np(dp - dt, w_gene, eps=eps)

    if baseline_wmae_per is None:
        base_wmae_per = weighted_mae_np(dt, w_gene, eps=eps)  # baseline pred=0
    else:
        base_wmae_per = np.asarray(baseline_wmae_per, dtype=np.float32)
        assert base_wmae_per.shape == (dt.shape[0],)

    ratio_per = pred_wmae_per / np.maximum(base_wmae_per, eps)

    improvement = 1.0 - ratio_per
    gate = gate_smoothstep_np(improvement, a=ratio_gate_a, b=ratio_gate_b)

    log_term = -np.log2(np.clip(ratio_per, eps, None))
    if log_clip is not None:
        log_term = np.clip(log_term, -float(log_clip), float(log_clip)).astype(np.float32)

    term_per = gate * log_term
    score_per = wcos_per * term_per

    out = {
        "wcos": wcos_per,
        "pred_wmae": pred_wmae_per,
        "base_wmae": base_wmae_per,
        "ratio": ratio_per,
        "gate": gate,
        "log_term": log_term,
        "term": term_per,
        "score": score_per,
    }
    return out

def per_pert_table(
    pert_ids,           # (N,) array-like
    comps: dict,        # from metric_components_per_row
    baseline_wmae=None  # optional (N,) from CSV, included for display
):
    pert_ids = np.asarray(pert_ids)
    df = pd.DataFrame({
        "pert_id": pert_ids,
        "score": comps["score"],
        "wcos": comps["wcos"],
        "pred_wmae": comps["pred_wmae"],
        "base_wmae": comps["base_wmae"],
        "ratio": comps["ratio"],
        "term": comps["term"],
        "gate": comps["gate"],
    })
    if baseline_wmae is not None:
        df["baseline_wmae_csv"] = np.asarray(baseline_wmae, dtype=np.float32)

    agg = df.groupby("pert_id", as_index=False).agg({
        "score": "mean",
        "wcos": "mean",
        "pred_wmae": "mean",
        "base_wmae": "mean",
        "ratio": "mean",
        "term": "mean",
        "gate": "mean",
        **({"baseline_wmae_csv": "mean"} if "baseline_wmae_csv" in df.columns else {})
    })
    agg = agg.sort_values("score", ascending=True).reset_index(drop=True)
    return agg, df

def show_worst_best(agg, k=15):
    print("Worst perts:")
    display(agg.head(k))
    print("\nBest perts:")
    display(agg.tail(k).sort_values("score", ascending=False))

def compare_runs(
    pert_ids,
    delta_true,
    pred_a,
    pred_b,
    *,
    name_a="A",
    name_b="B",
    baseline_wmae_per=None,  # if you want denominator override
):
    ca = metric_components_per_row(delta_true, pred_a, baseline_wmae_per=baseline_wmae_per)
    cb = metric_components_per_row(delta_true, pred_b, baseline_wmae_per=baseline_wmae_per)

    agg_a, _ = per_pert_table(pert_ids, ca)
    agg_b, _ = per_pert_table(pert_ids, cb)

    m = agg_a.merge(agg_b, on="pert_id", suffixes=(f"_{name_a}", f"_{name_b}"))
    m["d_score"] = m[f"score_{name_b}"] - m[f"score_{name_a}"]
    m["d_wcos"]  = m[f"wcos_{name_b}"]  - m[f"wcos_{name_a}"]
    m["d_ratio"] = m[f"ratio_{name_b}"] - m[f"ratio_{name_a}"]
    m["d_term"]  = m[f"term_{name_b}"]  - m[f"term_{name_a}"]

    # overall summaries
    def mean(x): return float(np.mean(x))
    summary = {
        f"{name_a}_mean_score": mean(ca["score"]),
        f"{name_b}_mean_score": mean(cb["score"]),
        "delta_mean_score": mean(cb["score"]) - mean(ca["score"]),
        f"{name_a}_wcos": mean(ca["wcos"]),
        f"{name_b}_wcos": mean(cb["wcos"]),
        f"{name_a}_ratio": mean(ca["ratio"]),
        f"{name_b}_ratio": mean(cb["ratio"]),
        f"{name_a}_term": mean(ca["term"]),
        f"{name_b}_term": mean(cb["term"]),
    }

    m_sorted = m.sort_values("d_score", ascending=True).reset_index(drop=True)
    return summary, m_sorted

# Example usage:
# summary, perts_delta = compare_runs(pert_ids, Y_true, pred_old, pred_new, name_a="old", name_b="new")
# print(summary)
# display(perts_delta.head(20))   # perts that got worse the most
# display(perts_delta.tail(20))   # perts that improved the most

def fit_alpha_log_baseline(
    delta_true,
    delta_pred,
    baseline_wmae,   # (N,)
    *,
    eps=1e-8,
    c0_grid=None,
    c1_grid=None,
    clip_lo=0.0,
    clip_hi=1.0,
):
    dt = np.asarray(delta_true, dtype=np.float32)
    dp = np.asarray(delta_pred, dtype=np.float32)
    b  = np.asarray(baseline_wmae, dtype=np.float32)

    x = np.log(b + eps).astype(np.float32)

    if c0_grid is None:
        c0_grid = np.linspace(0.60, 0.95, 36)  # adjust as needed
    if c1_grid is None:
        c1_grid = np.linspace(-0.25, 0.25, 41)

    best = None
    best_score = -1e9

    # precompute components that only depend on dt and dp scale? unfortunately score depends on scaled dp, so brute grid
    for c0 in c0_grid:
        for c1 in c1_grid:
            alpha = np.clip(c0 + c1 * x, clip_lo, clip_hi).astype(np.float32)
            dp2 = dp * alpha[:, None]
            comps = metric_components_per_row(dt, dp2)
            s = float(np.mean(comps["score"]))
            if s > best_score:
                best_score = s
                best = (float(c0), float(c1))

    c0, c1 = best
    alpha = np.clip(c0 + c1 * x, clip_lo, clip_hi).astype(np.float32)
    return {"best_score": best_score, "c0": c0, "c1": c1, "alpha": alpha}

def apply_alpha_log_baseline(delta_pred, baseline_wmae, c0, c1, *, eps=1e-8, clip_lo=0.0, clip_hi=1.0):
    b = np.asarray(baseline_wmae, dtype=np.float32)
    x = np.log(b + eps).astype(np.float32)
    alpha = np.clip(c0 + c1 * x, clip_lo, clip_hi).astype(np.float32)
    return np.asarray(delta_pred, dtype=np.float32) * alpha[:, None], alpha

# Example:
# fit = fit_alpha_log_baseline(Y_true, pred_oof, baseline_wmae_csv)
# print(fit["best_score"], fit["c0"], fit["c1"])
# pred_cal, alpha_vec = apply_alpha_log_baseline(pred_oof, baseline_wmae_csv, fit["c0"], fit["c1"])

import torch

def weighted_cosine_torch(dt, dp, w, eps=1e-12):
    wa = w * dt
    wb = w * dp
    num = torch.sum(wa * wb, dim=1)
    da  = torch.sqrt(torch.sum(wa * wa, dim=1))
    db  = torch.sqrt(torch.sum(wb * wb, dim=1))
    denom = torch.clamp(da * db, min=eps)
    return num / denom  # (B,)

def hybrid_loss_rowweighted_l1_plus_cosine(
    dt, dp, baseline_wmae,
    *,
    lam=0.9,
    eps=1e-8,
    clamp_min=0.5,
    clamp_max=3.0,
    mode="inv_sqrt",
):
    # L1-like (rowweighted)
    l_l1 = weighted_l1_like_rowweighted(
        dt, dp, baseline_wmae,
        eps=eps,
        mode=mode,
        clamp_min=clamp_min,
        clamp_max=clamp_max,
    )

    # Cosine term (mean of 1 - wcos_per)
    w_gene = gate_smoothstep(torch.abs(dt), a=GATE_A, b=GATE_B)  # (B,G)
    wcos_per = weighted_cosine_torch(dt, dp, w_gene, eps=1e-12)
    l_cos = torch.mean(1.0 - wcos_per)

    return lam * l_l1 + (1.0 - lam) * l_cos

def run_sweep(configs, fit_eval_fn):
    """
    configs: list[dict] each dict is hyperparams for one run
    fit_eval_fn(cfg) -> dict with:
        - "cv_mean"
        - "oof_score"
        - "fold_scores" list
        - optional: "notes"
    """
    rows = []
    for i, cfg in enumerate(configs):
        out = fit_eval_fn(cfg)
        row = {**cfg, **out}
        rows.append(row)
        print(f"[{i+1}/{len(configs)}] cfg={cfg}  cv_mean={out.get('cv_mean')}  oof={out.get('oof_score')}")
    return pd.DataFrame(rows)

# Example config lists:
def make_rank_sweep(R):
    return [{"RANK_R": r} for r in [max(1, R//2), R, 2*R]]

def make_dropout_sweep():
    return [{"DROPOUT": d} for d in [0.0, 0.05, 0.1, 0.2]]

def make_lr_sweep(base_lr):
    return [{"LR": x} for x in [base_lr * 0.8, base_lr, base_lr * 1.2]]


In [37]:
# ---- inputs you already have ----
dt = Y.astype(np.float32)              # (N, G)
dp_raw = oof_pred.astype(np.float32)   # (N, G)
baseline = delta_baseline.astype(np.float32)  # (G,)
alpha = float(best_global_alpha)       # 0.778

# baseline_wmae per row from your torch tensor
baseline_wmae = baseline_wmae_t.detach().cpu().numpy().astype(np.float32)  # (N,)

# Apply your exact shrink
dp = apply_shrink(dp_raw, baseline, alpha).astype(np.float32)  # (N, G)

# Metric components (uses your gate structure)
comps = metric_components_per_row(dt, dp)

print("OOF mean score:", float(np.mean(comps["score"])))
print("OOF mean wcos:", float(np.mean(comps["wcos"])))
print("OOF mean pred_wmae:", float(np.mean(comps["pred_wmae"])))
print("OOF mean base_wmae:", float(np.mean(comps["base_wmae"])))
print("OOF mean ratio:", float(np.mean(comps["ratio"])))
print("OOF mean term:", float(np.mean(comps["term"])))


OOF mean score: 0.16599032282829285
OOF mean wcos: 0.44331398606300354
OOF mean pred_wmae: 0.07494456321001053
OOF mean base_wmae: 0.09321802854537964
OOF mean ratio: 0.7897062301635742
OOF mean term: 0.3294218182563782


In [38]:
# comps from metric_components_per_row(dt, dp)
wcos_mean = float(np.mean(comps["wcos"]))
term_mean = float(np.mean(comps["term"]))
score_like_official = wcos_mean * term_mean

print("mean(wcos):", wcos_mean)
print("mean(term):", term_mean)
print("official-like score (mean(wcos)*mean(term)):", score_like_official)

print("what you printed before (mean of per-row products):", float(np.mean(comps["score"])))


mean(wcos): 0.44331398606300354
mean(term): 0.3294218182563782
official-like score (mean(wcos)*mean(term)): 0.14603729934735732
what you printed before (mean of per-row products): 0.16599032282829285


In [39]:
def official_like_score(dt, dp):
    comps = metric_components_per_row(dt, dp)
    return float(np.mean(comps["wcos"])) * float(np.mean(comps["term"]))

def fit_alpha_log_baseline_OFFICIAL(
    dt, dp_raw, baseline_vec, baseline_wmae,
    *,
    eps=1e-8,
    c0_grid=None,
    c1_grid=None,
    clip_lo=0.0,
    clip_hi=1.0,
):
    dt = np.asarray(dt, dtype=np.float32)
    dp_raw = np.asarray(dp_raw, dtype=np.float32)
    baseline_vec = np.asarray(baseline_vec, dtype=np.float32)
    bw = np.asarray(baseline_wmae, dtype=np.float32)
    x = np.log(bw + eps).astype(np.float32)

    if c0_grid is None:
        c0_grid = np.linspace(0.60, 0.95, 36)
    if c1_grid is None:
        c1_grid = np.linspace(-0.25, 0.25, 41)

    best = None
    best_score = -1e18

    for c0 in c0_grid:
        for c1 in c1_grid:
            alpha = np.clip(c0 + c1 * x, clip_lo, clip_hi).astype(np.float32)
            dp = dp_raw * alpha[:, None] + (1.0 - alpha)[:, None] * baseline_vec[None, :]
            sc = official_like_score(dt, dp)
            if sc > best_score:
                best_score = sc
                best = (float(c0), float(c1))

    c0, c1 = best
    alpha = np.clip(c0 + c1 * x, clip_lo, clip_hi).astype(np.float32)
    dp_best = dp_raw * alpha[:, None] + (1.0 - alpha)[:, None] * baseline_vec[None, :]
    return {"best_score": best_score, "c0": c0, "c1": c1, "alpha": alpha, "dp": dp_best}

# Run it
fit2 = fit_alpha_log_baseline_OFFICIAL(dt, dp_raw, baseline, baseline_wmae)
print("best official-like score:", fit2["best_score"])
print("c0, c1:", fit2["c0"], fit2["c1"])


best official-like score: 0.14628358660936325
c0, c1: 0.95 0.0625


In [40]:
import pandas as pd

agg = pd.read_csv("oof_per_pert_metrics.csv")

# Heuristic flags
bad_direction = agg[agg["wcos"] < 0.2].sort_values("wcos")
bad_ratio = agg[agg["ratio"] > 0.9].sort_values("ratio", ascending=False)

print("Direction problems (low wcos):")
display(bad_direction[["pert_id","score","wcos","ratio","term","baseline_wmae_csv"]].head(20))

print("Ratio problems (ratio>0.9):")
display(bad_ratio[["pert_id","score","wcos","ratio","term","baseline_wmae_csv"]].head(20))


Direction problems (low wcos):


,pert_id,score,wcos,ratio,term,baseline_wmae_csv
0,FLNA,-0.033154,-0.111312,0.811771,0.297851,0.043623
1,KAT2A,0.000164,0.078513,0.972412,0.002092,0.084314
8,SMARCC2,0.019634,0.120977,0.863107,0.162295,0.092904
19,COX5A,0.052843,0.155047,0.789593,0.340820,0.080229
2,USP22,0.004868,0.176639,0.931720,0.027557,0.098115
5,ACLY,0.013689,0.196193,0.903277,0.069774,0.171164


Ratio problems (ratio>0.9):


,pert_id,score,wcos,ratio,term,baseline_wmae_csv
1,KAT2A,0.000164,0.078513,0.972412,0.002092,0.084314
2,USP22,0.004868,0.176639,0.931720,0.027557,0.098115
3,STAG2,0.007683,0.227971,0.926491,0.033702,0.264928
4,PAGR1,0.009898,0.215542,0.917533,0.045923,0.180351
6,IL6ST,0.014940,0.320733,0.917092,0.046580,0.060349
7,PAXIP1,0.016088,0.258902,0.907509,0.062138,0.209861
9,INO80,0.024031,0.348380,0.903705,0.068980,0.165106
5,ACLY,0.013689,0.196193,0.903277,0.069774,0.171164


In [41]:
import numpy as np

# Z per pert. Use the same ordering as your training rows.
Z = Zt.detach().cpu().numpy().astype(np.float32)   # (N, d)
Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)

# pull from your per-pert table
bad_list = agg.sort_values("score").head(12)["pert_id"].tolist()

def top_neighbors(i, k=6):
    sims = Z @ Z[i]
    idx = np.argsort(-sims)
    return idx[1:k+1], sims[idx[1:k+1]]

# Map pert_id -> row index
# This assumes 1 row per pert. If your pert_ids has duplicates, adjust to pick the first.
pert_to_idx = {p:i for i,p in enumerate(gene_columns)}

for p in bad_list:
    i = pert_to_idx[p]
    nn_idx, nn_sim = top_neighbors(i, k=6)
    print("\n=== ", p, " ===")
    print("self score:", float(agg.loc[agg.pert_id==p, "score"].values[0]))
    print("neighbors:")
    for j, s in zip(nn_idx, nn_sim):
        pj = gene_columns[j]
        sj = float(agg.loc[agg.pert_id==pj, "score"].values[0])
        print(f"  {pj:10s}  sim={float(s):.3f}  score={sj:.4f}")


IndexError: index 1631 is out of bounds for axis 0 with size 80

LR = 0.0018
fold 1: best_score=0.166176 best_alpha=0.778 best_epoch=31
fold 2: best_score=0.114720 best_alpha=0.778 best_epoch=28
fold 3: best_score=0.113649 best_alpha=0.778 best_epoch=25
fold 4: best_score=0.122409 best_alpha=0.778 best_epoch=43
fold 5: best_score=0.158052 best_alpha=0.778 best_epoch=26
fold 6: best_score=0.191780 best_alpha=0.778 best_epoch=24
fold 7: best_score=0.182183 best_alpha=0.778 best_epoch=16
fold 8: best_score=0.129139 best_alpha=0.778 best_epoch=18
cv mean: 0.14726350865234605 std: 0.0291769675525368
median best_epoch = 25
OOF global alpha: 0.778 OOF score: 0.14603729934735732

In [32]:
# Replace this with your real pert id column
# Example: pert_ids = train_df["pert_id"].to_numpy()
pert_ids = train_genes # must be length N

agg, row_df = per_pert_table(pert_ids, comps, baseline_wmae=baseline_wmae)
show_worst_best(agg, k=20)

# Save it
agg.to_csv("oof_per_pert_metrics.csv", index=False)
print("[ok] wrote oof_per_pert_metrics.csv")


Worst perts:


,pert_id,score,wcos,pred_wmae,base_wmae,ratio,term,gate,baseline_wmae_csv
0,FLNA,-0.033154,-0.111312,0.045976,0.056636,0.811771,0.297851,0.990015,0.043623
1,KAT2A,0.000164,0.078513,0.073288,0.075368,0.972412,0.002092,0.051835,0.084314
2,USP22,0.004868,0.176639,0.090229,0.096842,0.931720,0.027557,0.270081,0.098115
3,STAG2,0.007683,0.227971,0.148434,0.160211,0.926491,0.033702,0.305963,0.264928
4,PAGR1,0.009898,0.215542,0.106261,0.115811,0.917533,0.045923,0.369849,0.180351
5,ACLY,0.013689,0.196193,0.114734,0.127020,0.903277,0.069774,0.475432,0.171164
6,IL6ST,0.014940,0.320733,0.055904,0.060958,0.917093,0.046580,0.373055,0.060349
7,PAXIP1,0.016088,0.258902,0.119392,0.131560,0.907509,0.062138,0.443792,0.209861
8,SMARCC2,0.019634,0.120977,0.070822,0.082054,0.863107,0.162295,0.764142,0.092904
9,INO80,0.024031,0.348381,0.120555,0.133401,0.903705,0.068980,0.472223,0.165106



Best perts:


,pert_id,score,wcos,pred_wmae,base_wmae,ratio,term,gate,baseline_wmae_csv
79,PRCP,0.490046,0.660412,0.036021,0.060245,0.597897,0.742031,1.0,0.054376
78,PMEL,0.455713,0.675065,0.037834,0.060408,0.626304,0.675065,1.0,0.049146
77,APAF1,0.439850,0.627243,0.039476,0.064185,0.615042,0.701243,1.0,0.107923
76,VEGFA,0.411820,0.700933,0.036965,0.055547,0.665481,0.587531,1.0,0.048473
75,SUPT4H1,0.400396,0.722216,0.059996,0.088107,0.680941,0.554399,1.0,0.083862
74,DZIP3,0.377061,0.570530,0.039288,0.062118,0.632485,0.660896,1.0,0.072408
73,KDM4A,0.362451,0.591003,0.037102,0.056757,0.653708,0.613281,1.0,0.049926
72,KAT8,0.341303,0.680790,0.051934,0.073513,0.706454,0.501333,1.0,0.065785
71,CHMP4B,0.336462,0.633826,0.050799,0.073394,0.692151,0.530842,1.0,0.065029
70,DNAJA3,0.316319,0.672337,0.063972,0.088637,0.721726,0.470477,1.0,0.083669


[ok] wrote oof_per_pert_metrics.csv


In [33]:
tmp = pd.DataFrame({
    "score": comps["score"],
    "wcos": comps["wcos"],
    "ratio": comps["ratio"],
    "term": comps["term"],
    "baseline_wmae": baseline_wmae,
})

print(tmp.corr(numeric_only=True))

# Quick buckets by baseline_wmae to see where you're weak
tmp["bw_bin"] = pd.qcut(tmp["baseline_wmae"], q=10, duplicates="drop")
bucket = tmp.groupby("bw_bin", as_index=False).agg({
    "score": "mean",
    "wcos": "mean",
    "ratio": "mean",
    "term": "mean",
    "baseline_wmae": "mean",
})
display(bucket)


                  score      wcos     ratio      term  baseline_wmae
score          1.000000  0.868312 -0.933108  0.929605      -0.377670
wcos           0.868312  1.000000 -0.715972  0.707081      -0.224055
ratio         -0.933108 -0.715972  1.000000 -0.997542       0.399736
term           0.929605  0.707081 -0.997542  1.000000      -0.410573
baseline_wmae -0.377670 -0.224055  0.399736 -0.410573       1.000000


C:\Users\rowes\AppData\Local\Temp\ipykernel_8648\2794997053.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket = tmp.groupby("bw_bin", as_index=False).agg({


,bw_bin,score,wcos,ratio,term,baseline_wmae
0,"(0.0426, 0.0598]",0.283642,0.518995,0.713176,0.496503,0.050613
1,"(0.0598, 0.0734]",0.234724,0.514146,0.744758,0.423140,0.064322
2,"(0.0734, 0.0839]",0.246115,0.511683,0.724990,0.465267,0.079760
3,"(0.0839, 0.0955]",0.127344,0.370428,0.812269,0.292138,0.089246
4,"(0.0955, 0.104]",0.153359,0.435117,0.800340,0.305946,0.100619
5,"(0.104, 0.112]",0.230291,0.532122,0.748974,0.418589,0.106864
6,"(0.112, 0.128]",0.175897,0.477288,0.781040,0.350680,0.118385
7,"(0.128, 0.155]",0.077636,0.361831,0.849921,0.198575,0.140536
8,"(0.155, 0.199]",0.068205,0.345329,0.852768,0.188078,0.171342
9,"(0.199, 0.633]",0.062691,0.366200,0.868825,0.155303,0.345952


LR = 2e-3
fold 1: best_score=0.162418 best_alpha=0.700 best_epoch=25
fold 2: best_score=0.111889 best_alpha=0.700 best_epoch=25
fold 3: best_score=0.114223 best_alpha=0.680 best_epoch=25
fold 4: best_score=0.121005 best_alpha=0.700 best_epoch=25
fold 5: best_score=0.156279 best_alpha=0.700 best_epoch=25
fold 6: best_score=0.188296 best_alpha=0.700 best_epoch=25
fold 7: best_score=0.178646 best_alpha=0.640 best_epoch=25
fold 8: best_score=0.123527 best_alpha=0.620 best_epoch=25
cv mean: 0.1445352986908065 std: 0.028539096177756983
median best_epoch = 25
OOF global alpha: 0.699999988079071 OOF score: 0.14320898877057076

0.6 * 2e-3
fold 1: best_score=0.163546 best_alpha=0.700 best_epoch=25
fold 2: best_score=0.113625 best_alpha=0.700 best_epoch=25
fold 3: best_score=0.114235 best_alpha=0.700 best_epoch=25
fold 4: best_score=0.120765 best_alpha=0.700 best_epoch=50
fold 5: best_score=0.155355 best_alpha=0.700 best_epoch=25
fold 6: best_score=0.190968 best_alpha=0.700 best_epoch=25
fold 7: best_score=0.181645 best_alpha=0.680 best_epoch=25
fold 8: best_score=0.127556 best_alpha=0.680 best_epoch=25
cv mean: 0.14596182446285866 std: 0.028987465686809547
median best_epoch = 25
OOF global alpha: 0.699999988079071 OOF score: 0.14478725199154052

0.5 * 2e-3
fold 1: best_score=0.163336 best_alpha=0.700 best_epoch=50
fold 2: best_score=0.113405 best_alpha=0.700 best_epoch=50
fold 3: best_score=0.113684 best_alpha=0.700 best_epoch=25
fold 4: best_score=0.120943 best_alpha=0.700 best_epoch=50
fold 5: best_score=0.154866 best_alpha=0.700 best_epoch=75
fold 6: best_score=0.188617 best_alpha=0.700 best_epoch=25
fold 7: best_score=0.182520 best_alpha=0.700 best_epoch=25
fold 8: best_score=0.128315 best_alpha=0.700 best_epoch=25
cv mean: 0.14571076563126395 std: 0.028670166220607033
median best_epoch = 37
OOF global alpha: 0.699999988079071 OOF score: 0.14454178038006305

0.25 * 2e-3
fold 1: best_score=0.162586 best_alpha=0.700 best_epoch=75
fold 2: best_score=0.112990 best_alpha=0.700 best_epoch=50
fold 3: best_score=0.113660 best_alpha=0.680 best_epoch=75
fold 4: best_score=0.120988 best_alpha=0.700 best_epoch=100
fold 5: best_score=0.155519 best_alpha=0.700 best_epoch=75
fold 6: best_score=0.189735 best_alpha=0.700 best_epoch=75
fold 7: best_score=0.182503 best_alpha=0.700 best_epoch=50
fold 8: best_score=0.127621 best_alpha=0.700 best_epoch=50
cv mean: 0.14570035636447376 std: 0.028959364164949458
median best_epoch = 75
OOF global alpha: 0.699999988079071 OOF score: 0.1445417741413877

GATE_B = 0.20

fold 1: best_score=0.162418 best_alpha=0.700 best_epoch=25
fold 2: best_score=0.111889 best_alpha=0.700 best_epoch=25
fold 3: best_score=0.114223 best_alpha=0.680 best_epoch=25
fold 4: best_score=0.121005 best_alpha=0.700 best_epoch=25
fold 5: best_score=0.156279 best_alpha=0.700 best_epoch=25
fold 6: best_score=0.188296 best_alpha=0.700 best_epoch=25
fold 7: best_score=0.178646 best_alpha=0.640 best_epoch=25
fold 8: best_score=0.123527 best_alpha=0.620 best_epoch=25
cv mean: 0.1445352986908065 std: 0.028539096177756983
median best_epoch = 25
OOF global alpha: 0.699999988079071 OOF score: 0.14320898877057076

GATE_B = 0.15

fold 1: best_score=0.156727 best_alpha=0.700 best_epoch=25
fold 2: best_score=0.111402 best_alpha=0.700 best_epoch=25
fold 3: best_score=0.114206 best_alpha=0.700 best_epoch=25
fold 4: best_score=0.117499 best_alpha=0.700 best_epoch=25
fold 5: best_score=0.150786 best_alpha=0.700 best_epoch=25
fold 6: best_score=0.187015 best_alpha=0.700 best_epoch=25
fold 7: best_score=0.181046 best_alpha=0.700 best_epoch=25
fold 8: best_score=0.123531 best_alpha=0.700 best_epoch=25
cv mean: 0.14277655599062244 std: 0.02848315064548624
median best_epoch = 25
OOF global alpha: 0.699999988079071 OOF score: 0.14165771847855702

GATE_B = 0.25

fold 1: best_score=0.165125 best_alpha=0.700 best_epoch=25
fold 2: best_score=0.111574 best_alpha=0.700 best_epoch=25
fold 3: best_score=0.113811 best_alpha=0.640 best_epoch=25
fold 4: best_score=0.122367 best_alpha=0.700 best_epoch=25
fold 5: best_score=0.157754 best_alpha=0.700 best_epoch=25
fold 6: best_score=0.187647 best_alpha=0.660 best_epoch=25
fold 7: best_score=0.175402 best_alpha=0.600 best_epoch=25
fold 8: best_score=0.122233 best_alpha=0.580 best_epoch=25
cv mean: 0.1444892159128771 std: 0.028348884529086005
median best_epoch = 25
OOF global alpha: 0.6800000071525574 OOF score: 0.14281007902427767

loss = weighted_l1_like_rowweighted(
                dt_b, dp_b, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

fold 1: best_score=0.162418 best_alpha=0.700 best_epoch=25
fold 2: best_score=0.111889 best_alpha=0.700 best_epoch=25
fold 3: best_score=0.114223 best_alpha=0.680 best_epoch=25
fold 4: best_score=0.121005 best_alpha=0.700 best_epoch=25
fold 5: best_score=0.156279 best_alpha=0.700 best_epoch=25
fold 6: best_score=0.188296 best_alpha=0.700 best_epoch=25
fold 7: best_score=0.178646 best_alpha=0.640 best_epoch=25
fold 8: best_score=0.123527 best_alpha=0.620 best_epoch=25
cv mean: 0.1445352986908065 std: 0.028539096177756983
median best_epoch = 25
OOF global alpha: 0.699999988079071 OOF score: 0.14320898877057076

loss = weighted_l1_like_rowweighted(
                dt_b, dp_b, bw_b,
                mode="inv_sqrt",
                clamp_min=0.7,
                clamp_max=2.0,
)

fold 1: best_score=0.163247 best_alpha=0.700 best_epoch=25
fold 2: best_score=0.112429 best_alpha=0.700 best_epoch=25
fold 3: best_score=0.111956 best_alpha=0.640 best_epoch=25
fold 4: best_score=0.122694 best_alpha=0.700 best_epoch=25
fold 5: best_score=0.157536 best_alpha=0.700 best_epoch=25
fold 6: best_score=0.187481 best_alpha=0.680 best_epoch=25
fold 7: best_score=0.175097 best_alpha=0.620 best_epoch=25
fold 8: best_score=0.123204 best_alpha=0.620 best_epoch=25
cv mean: 0.14420547200856515 std: 0.02810893605355988
median best_epoch = 25
OOF global alpha: 0.699999988079071 OOF score: 0.1427402023207307

loss = weighted_l1_like_rowweighted(
                dt_b, dp_b, bw_b,
                mode="inv_sqrt",
                clamp_min=0.3,
                clamp_max=5.0,
            )

fold 1: best_score=0.159718 best_alpha=0.700 best_epoch=25
fold 2: best_score=0.111553 best_alpha=0.700 best_epoch=25
fold 3: best_score=0.115490 best_alpha=0.700 best_epoch=25
fold 4: best_score=0.118156 best_alpha=0.700 best_epoch=25
fold 5: best_score=0.151292 best_alpha=0.700 best_epoch=25
fold 6: best_score=0.189359 best_alpha=0.700 best_epoch=25
fold 7: best_score=0.179760 best_alpha=0.660 best_epoch=25
fold 8: best_score=0.126236 best_alpha=0.660 best_epoch=25
cv mean: 0.14394549028394532 std: 0.028472675748411735
median best_epoch = 25
OOF global alpha: 0.699999988079071 OOF score: 0.14277189015750835

loss = weighted_l1_like_rowweighted(
                dt_b, dp_b, bw_b,
                mode="inv_log",
                clamp_min=0.5,
                clamp_max=3.0,
)

fold 1: best_score=0.163073 best_alpha=0.700 best_epoch=25
fold 2: best_score=0.112248 best_alpha=0.700 best_epoch=25
fold 3: best_score=0.111654 best_alpha=0.640 best_epoch=25
fold 4: best_score=0.123021 best_alpha=0.700 best_epoch=25
fold 5: best_score=0.157770 best_alpha=0.700 best_epoch=25
fold 6: best_score=0.187842 best_alpha=0.680 best_epoch=25
fold 7: best_score=0.174498 best_alpha=0.620 best_epoch=25
fold 8: best_score=0.123217 best_alpha=0.620 best_epoch=25
cv mean: 0.14416549925900013 std: 0.028133274757033498
median best_epoch = 25
OOF global alpha: 0.699999988079071 OOF score: 0.14268128816927295

loss = weighted_l1_like_rowweighted(
                dt_b, dp_b, bw_b,
                mode="inv",
                clamp_min=0.5,
                clamp_max=3.0,
)

fold 1: best_score=0.163200 best_alpha=0.700 best_epoch=25
fold 2: best_score=0.112736 best_alpha=0.700 best_epoch=25
fold 3: best_score=0.112199 best_alpha=0.640 best_epoch=25
fold 4: best_score=0.122677 best_alpha=0.700 best_epoch=25
fold 5: best_score=0.157525 best_alpha=0.700 best_epoch=25
fold 6: best_score=0.187082 best_alpha=0.680 best_epoch=25
fold 7: best_score=0.175008 best_alpha=0.620 best_epoch=25
fold 8: best_score=0.123332 best_alpha=0.620 best_epoch=25
cv mean: 0.1442199672741995 std: 0.027927069221901964
median best_epoch = 25
OOF global alpha: 0.699999988079071 OOF score: 0.14277206537857978

In [ ]:

def fit_full_model(seed, epochs_fixed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    for epoch in range(1, epochs_fixed + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            dt_b = Yt.index_select(0, b_t)                 # (B, G)
            bw_b = baseline_wmae_t.index_select(0, b_t)     # (B,)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == epochs_fixed:
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt, Uo_t).detach().cpu().numpy().astype(np.float32)

            pred_np = apply_shrink(pred_np, delta_baseline, ALPHA_SHRINK)
            s = score_delta(Y, pred_np)
            print(f"[seed {seed}] epoch={epoch:4d} train_score={s['score']:.6f} wcos={s['wcos']:.6f} pred_wmae={s['pred_wmae']:.6f} alpha={ALPHA_SHRINK:.3f}")

    model.eval()
    return model

# refit ensemble on ALL 80 perts, using CV-calibrated epoch + OOF-calibrated alpha
models = [fit_full_model(sd, epochs_fixed=EPOCHS_MED) for sd in MODEL_SEEDS]
print("Refit models:", len(models))

def predict_delta_gene(gene_symbol: str) -> np.ndarray:
    z = torch.tensor(emb_pert(gene_symbol)[None, :].astype(np.float32), device=device)
    preds = []
    with torch.no_grad():
        for m in models:
            y = m(z, Uo_t).detach().cpu().numpy().astype(np.float32)[0]
            preds.append(y)
    yhat = np.mean(np.stack(preds, axis=0), axis=0).astype(np.float32)
    yhat = (ALPHA_SHRINK * yhat + (1.0 - ALPHA_SHRINK) * delta_baseline).astype(np.float32)
    return yhat

# Build submission from sample_submission.csv
sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)
sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

# ensure order matches gene_columns
idx = {g: i for i, g in enumerate(gene_columns)}
perm = [idx[g] for g in sub_gene_cols]

# fill default baseline for unknown test perts
sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

# fill known val perts (pert_1..pert_60)
hit = 0
for pid, gene in val_map.items():
    vec = predict_delta_gene(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

out_path = "submission_bilinear_refit_oofalpha.csv"
sub.to_csv(out_path, index=False)
print("[ok] wrote:", out_path, "| filled:", hit)


[seed 6] epoch=  25 train_score=0.191764 wcos=0.490486 pred_wmae=0.072424 alpha=0.778
[seed 6] epoch=  27 train_score=0.192631 wcos=0.490842 pred_wmae=0.072345 alpha=0.778
[seed 7] epoch=  25 train_score=0.192178 wcos=0.490857 pred_wmae=0.072399 alpha=0.778
[seed 7] epoch=  27 train_score=0.192619 wcos=0.490718 pred_wmae=0.072349 alpha=0.778
[seed 8] epoch=  25 train_score=0.191820 wcos=0.490593 pred_wmae=0.072431 alpha=0.778
[seed 8] epoch=  27 train_score=0.192701 wcos=0.490830 pred_wmae=0.072347 alpha=0.778
Refit models: 3
[ok] wrote: submission_bilinear_refit_oofalpha.csv | filled: 60


[seed 6] epoch=  25 train_score=0.190763 wcos=0.493199 pred_wmae=0.072467 alpha=0.700
[seed 7] epoch=  25 train_score=0.191113 wcos=0.493597 pred_wmae=0.072458 alpha=0.700
[seed 8] epoch=  25 train_score=0.190862 wcos=0.493393 pred_wmae=0.072469 alpha=0.700
Refit models: 3
[ok] wrote: submission_bilinear_refit_oofalpha.csv | filled: 60

epoch=  25 train_score=0.197565 wcos=0.487435 pred_wmae=0.071318
epoch=  50 train_score=0.214566 wcos=0.506001 pred_wmae=0.070564
epoch=  75 train_score=0.255586 wcos=0.546834 pred_wmae=0.068880
epoch= 100 train_score=0.320180 wcos=0.596986 pred_wmae=0.066354
epoch= 125 train_score=0.395301 wcos=0.641472 pred_wmae=0.063575
epoch= 150 train_score=0.465017 wcos=0.673950 pred_wmae=0.061027
epoch= 175 train_score=0.528761 wcos=0.699482 pred_wmae=0.058800
epoch= 200 train_score=0.583847 wcos=0.718520 pred_wmae=0.056899
epoch= 225 train_score=0.634970 wcos=0.734100 pred_wmae=0.055180
epoch= 250 train_score=0.678502 wcos=0.748007 pred_wmae=0.053726
epoch= 275 train_score=0.714378 wcos=0.758516 pred_wmae=0.052467
epoch= 300 train_score=0.750164 wcos=0.767519 pred_wmae=0.051263
epoch= 325 train_score=0.780506 wcos=0.776008 pred_wmae=0.050237
epoch= 350 train_score=0.811266 wcos=0.784896 pred_wmae=0.049284
epoch= 375 train_score=0.837425 wcos=0.791671 pred_wmae=0.048401
epoch= 400 train_score=0.859289 wcos=0.797939 pred_wmae=0.047662
wrote: submission_bilinear.csv